# Local path planning

Inspect quintic lateral candidates and explicit rejection reasons.

Run with the repository virtual environment. These are synthetic research experiments; units and assumptions are explicit below.

## Candidate configuration

This sampled planner uses point clearance with a conservative margin. It is separate from the simulator driver.

In [ ]:
import matplotlib.pyplot as plt

from autonomous_vehicles.planning.lattice import LatticeConfig, LatticePlanner
from autonomous_vehicles.planning.polynomial import Boundary, Quintic
from autonomous_vehicles.simulation.geometry import Pose2
from autonomous_vehicles.simulation.roads import straight_road
from autonomous_vehicles.simulation.shapes import Rectangle
from autonomous_vehicles.visualization import plot_candidates

road = straight_road(width=10)
planner = LatticePlanner(LatticeConfig(clearance=1.0))

## Smooth transition

The quintic matches position, first derivative and second derivative at both endpoints.

In [ ]:
polynomial = Quintic(Boundary(0), Boundary(2), 30)
coordinates, lateral = polynomial.sample()
fig, axes = plt.subplots(1, 3, figsize=(12, 3))
for derivative, ax in enumerate(axes):
    ax.plot(coordinates, polynomial.value(coordinates, derivative))
    ax.set_xlabel("Longitudinal distance (m)")
    ax.set_title(f"Derivative order {derivative}")
plt.show()
print("Integrated squared third derivative:", polynomial.integrated_squared_jerk())

## Unobstructed choice

With symmetric costs and no obstacles, the center candidate has the smallest deviation cost.

In [ ]:
candidates = planner.candidates(road, start_s=5)
selected = planner.select(candidates)
plot_candidates(candidates, road)
plt.show()
print("Selected offset:", selected.offset if selected else None)
assert selected is not None and selected.offset == 0
for candidate in candidates:
    print(candidate.offset, candidate.cost, candidate.reason)

## Obstacle rejection

Changing a constraint changes feasibility, not just a plotted line.

In [ ]:
obstacles = (Rectangle(Pose2(25, 0), 3, 1.5),)
blocked = planner.candidates(road, 5, obstacles=obstacles)
plot_candidates(blocked, road)
plt.show()
for candidate in blocked:
    print(
        {
            "offset": candidate.offset,
            "reason": candidate.reason,
            "minimum_clearance_m": candidate.minimum_clearance,
        }
    )
selected = planner.select(blocked)
print("Feasible choice:", selected.offset if selected else None)

## No feasible path

An entirely blocked corridor returns None; the caller must handle stopping or replanning.

In [ ]:
wall = Rectangle(Pose2(20, 0), 5, 15)
rejected = planner.candidates(road, 5, obstacles=(wall,))
assert planner.select(rejected) is None
print([candidate.reason for candidate in rejected])
fig, ax = plt.subplots(figsize=(6, 3))
ax.bar([str(c.offset) for c in candidates], [c.cost for c in candidates])
ax.set_xlabel("Terminal lateral offset (m)")
ax.set_ylabel("Candidate cost")
plt.show()

## Research record

Record the seed, configuration, dependency lock and observed failures with each experiment. Results here demonstrate the implemented model under its stated assumptions. They do not establish performance outside this synthetic distribution.